In [1]:
import mmcv
from mmengine import Config
# Check Pytorch installation
import torch, torchvision
print("torch version:",torch.__version__, "cuda:",torch.cuda.is_available())
# Check MMDetection installation
import mmdet
print("mmdetection:",mmdet.__version__)
# Check mmcv installation
import mmcv
print("mmcv:",mmcv.__version__)
# Check mmengine installation
import mmengine
print("mmengine:",mmengine.__version__)



torch version: 2.1.0+cu121 cuda: True
mmdetection: 3.3.0
mmcv: 2.1.0
mmengine: 0.10.5


In [3]:
from mmengine.runner import set_random_seed
import os
CONFIG = './configs/ms_rcnn/ms-rcnn_x101-64x4d_fpn_2x_coco.py'
config = CONFIG[:-3] + "-Son.py"

if os.path.isfile(config):
    print("True")
    os.remove(config)

cfg = Config.fromfile(CONFIG)
# Modify dataset classes and color
cfg.metainfo = {
    'classes': ('cell', ),
    'palette': [
        (220, 20, 60),
    ]
}



# Modify dataset type and path
cfg.data_root = './coco'

cfg.train_dataloader.dataset.ann_file = 'train/train.json'
cfg.train_dataloader.dataset.data_root = cfg.data_root
cfg.train_dataloader.dataset.data_prefix.img = 'train/images/'
cfg.train_dataloader.dataset.metainfo = cfg.metainfo
cfg.train_dataloader.batch_size = 8
cfg.train_dataloader.num_workers = 0
cfg.train_dataloader.pin_memory = True 
cfg.train_dataloader.persistent_workers = False

cfg.val_dataloader.dataset.ann_file = 'val/val.json'
cfg.val_dataloader.dataset.data_root = cfg.data_root
cfg.val_dataloader.dataset.data_prefix.img = 'val/images/'
cfg.val_dataloader.dataset.metainfo = cfg.metainfo
cfg.val_dataloader.batch_size = 8
cfg.val_dataloader.num_workers = 4
cfg.val_dataloader.pin_memory = True 

#cfg.test_dataloader = cfg.val_dataloader
cfg.test_dataloader.dataset.ann_file = 'test/test.json'
cfg.test_dataloader.dataset.data_root = cfg.data_root
cfg.test_dataloader.dataset.data_prefix.img = 'test/images/'
cfg.test_dataloader.dataset.metainfo = cfg.metainfo
cfg.val_dataloader.batch_size = 8
cfg.val_dataloader.num_workers = 4
cfg.val_dataloader.pin_memory = True 

# Modify metric config
cfg.val_evaluator.ann_file = cfg.data_root+'/'+'val/val.json'
cfg.test_evaluator.ann_file = cfg.data_root+'/'+'test/test.json'

# Modify num classes of the model in box head and mask head
cfg.model.roi_head.bbox_head.num_classes = 1
cfg.model.roi_head.mask_head.num_classes = 1

# We can still the pre-trained Mask RCNN model to obtain a higher performance
#cfg.load_from = 'checkpoints/mask_rcnn_r50_caffe_fpn_mstrain-poly_3x_coco_bbox_mAP-0.408__segm_mAP-0.37_20200504_163245-42aa3d00.pth'

# Set up working dir to save files and logs.
cfg.work_dir = CONFIG[:-3].split("/")[-1]


# We can set the evaluation interval to reduce the evaluation times
cfg.train_cfg.val_interval = 1
# We can set the checkpoint saving interval to reduce the storage cost
#cfg.default_hooks.checkpoint.interval = 1
cfg.train_cfg.max_epochs = 26
cfg.max_epochs = 26
# The original learning rate (LR) is set for 8-GPU training.
# We divide it by 8 since we only use one GPU.
cfg.optim_wrapper.optimizer.lr = 0.02 / 8
cfg.default_hooks.logger.interval = 500
cfg.default_hooks.checkpoint.interval = -1
cfg.default_hooks.checkpoint.save_best = 'coco/segm_mAP'
cfg.default_hooks.checkpoint.rule = "greater"

cfg.val_evaluator.metric = ["segm"]
# Set seed thus the results are more reproducible
cfg.seed = 0
set_random_seed(0, deterministic=False)

# We can also use tensorboard to log the training process
#cfg.visualizer.vis_backends.append({"type":'TensorboardVisBackend'})

#------------------------------------------------------
#config = f'./configs/mask_rcnn/mask-rcnn_x101-32x4d_fpn_1x_coco-Son.py'
with open(config, 'w+') as f:
    f.write(cfg.pretty_text)

True


In [5]:
#config = './configs/ms_rcnn/ms-rcnn_x101-32x4d_fpn_1x_coco-Son.py'
config

'./configs/ms_rcnn/ms-rcnn_x101-64x4d_fpn_2x_coco-Son.py'

In [7]:
!python tools/train.py './configs/ms_rcnn/ms-rcnn_x101-64x4d_fpn_2x_coco-Son.py'


Random seed set as 0
10/24 21:05:07 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.14 (main, May  6 2024, 19:42:50) [GCC 11.2.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 209652396
    GPU 0: NVIDIA GeForce RTX 4090
    CUDA_HOME: None
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
    PyTorch: 2.1.0+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;


KeyboardInterrupt



In [6]:
!rm -rf ms-rcnn_x101-64x4d_fpn_2x_coco

In [36]:
!python tools/train.py {config}

10/13 13:29:14 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.14 (main, May  6 2024, 19:42:50) [GCC 11.2.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1776855110
    GPU 0: NVIDIA GeForce RTX 4090
    CUDA_HOME: None
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
    PyTorch: 2.1.0+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;arch=compute_70,code

KeyboardInterrupt: 

In [39]:
!rm -rf ms-rcnn_x101-32x4d_fpn_1x_coco

In [7]:
!rm -rf ms-rcnn_r101-caffe_fpn_2x_coco

In [14]:
!python tools/test.py \
./ms-rcnn_r101-caffe_fpn_2x_coco/ms-rcnn_r101-caffe_fpn_2x_coco-Son.py \
./ms-rcnn_r101-caffe_fpn_2x_coco/best_coco_segm_mAP_epoch_18.pth

10/13 13:08:39 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.14 (main, May  6 2024, 19:42:50) [GCC 11.2.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1661718011
    GPU 0: NVIDIA GeForce RTX 4090
    CUDA_HOME: None
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
    PyTorch: 2.1.0+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;arch=compute_70,code

In [ ]:
!python -m torch.utils.bottleneck tools/train.py {config} 

`bottleneck` is a tool that can be used as an initial step for debugging
bottlenecks in your program.

It summarizes runs of your script with the Python profiler and PyTorch's
autograd profiler. Because your script will be profiled, please ensure that it
exits in a finite amount of time.

For more complicated uses of the profilers, please see
https://docs.python.org/3/library/profile.html and
https://pytorch.org/docs/master/autograd.html#profiler for more information.
Running environment analysis...
Running your script with cProfile
10/13 10:32:44 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.14 (main, May  6 2024, 19:42:50) [GCC 11.2.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1796385112
    GPU 0: NVIDIA GeForce RTX 4090
    CUDA_HOME: None
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
    PyTorch: 2.1.0+cu121
    PyTorch compiling details: PyTorch bu

`bottleneck` is a tool that can be used as an initial step for debugging
bottlenecks in your program.

It summarizes runs of your script with the Python profiler and PyTorch's
autograd profiler. Because your script will be profiled, please ensure that it
exits in a finite amount of time.

For more complicated uses of the profilers, please see
https://docs.python.org/3/library/profile.html and
https://pytorch.org/docs/master/autograd.html#profiler for more information.
Running environment analysis...
Running your script with cProfile
10/13 00:21:00 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.14 (main, May  6 2024, 19:42:50) [GCC 11.2.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 1935130230
    GPU 0: NVIDIA GeForce RTX 4090
    CUDA_HOME: None
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
    PyTorch: 2.1.0+cu121
    PyTorch compiling details: PyTorch bu

In [ ]:
python tools/train.py './configs/mask_rcnn/mask-rcnn_r101_fpn_2x_coco-Son.py'

In [5]:
!python tools/train.py {config}

10/09 21:50:13 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.10.14 (main, May  6 2024, 19:42:50) [GCC 11.2.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 446234222
    GPU 0: NVIDIA GeForce RTX 2080 Ti
    CUDA_HOME: None
    GCC: gcc (Ubuntu 11.4.0-1ubuntu1~22.04) 11.4.0
    PyTorch: 2.1.0+cu121
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.1.1 (Git Hash 64f6bcbcbab628e96f33a62c3e975f8535a7bde4)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;arch=compute_70,code

In [ ]:
python tools/test.py \
    ${config} \
    ${cfg.work_dir} \